# 🔬 Lab 05: Next-Gen Re-Ranking & Query Transformation

---

## 🎯 Executive Overview & Production Challenge
First-stage vector retrieval uses **Bi-Encoders** to compress whole chunks into single fixed-size vectors. This is fast ($O(\log N)$) but imprecise:
1. **The Vocabulary Mismatch Problem:** When user questions use different phrasing or colloquial terms than the source documents.
2. **Cross-Encoder vs Bi-Encoder:** A **Cross-Encoder** evaluates `(Query, Chunk)` simultaneously with full cross-attention across every word token ($O(L^2)$). It is much more accurate, but too slow to run across 1M docs.
3. **The Production Two-Stage Architecture:**
```
  [User Query]
       │
  Query Transformation (HyDE / Multi-Query Expansion)
       │
  Stage 1: Fast Broad Retrieval (Dense Vectors + BM25 Lexical) ──> Fetch Top 30
       │
  Reciprocal Rank Fusion (RRF) ──> Blend keyword and semantic ranks
       │
  Stage 2: Cross-Encoder Neural Re-Ranker (BAAI / MiniLM) ──> Re-score Top 30
       │
  Top 3 High-Precision Chunks to LLM Context Window
```


In [6]:
# Pre-flight imports & verification
import os
import sys
import re
import tempfile
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# Configure sandboxed cache directories for Torch, HuggingFace, and Matplotlib
temp_dir = tempfile.gettempdir()
os.environ.setdefault("MPLCONFIGDIR", os.path.join(temp_dir, "matplotlib"))
os.environ.setdefault("TORCH_HOME", os.path.join(temp_dir, "torch"))
os.environ.setdefault("HF_HOME", os.path.join(temp_dir, "hf"))
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", os.path.join(temp_dir, "torch_inductor"))

# Self-healing sys.path resolution so rag_universal_client is ALWAYS found
for candidate in [
    Path.cwd(),
    Path.cwd() / "RAG_ContxtEngg_Labs",
    Path(__file__).parent if "__file__" in locals() else None,
    Path.cwd() / "UST" / "RAG_ContxtEngg_Labs",
    Path(r"c:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs")
]:
    if candidate and candidate.exists() and (candidate / "rag_universal_client.py").exists():
        if str(candidate.resolve()) not in sys.path:
            sys.path.insert(0, str(candidate.resolve()))
        break
import rag_universal_client as rc

# Resilient path resolution for sample_documents
DATA_DIR = rc.get_rag_data_dir()
print(f"[*] Verified Data Directory: {DATA_DIR}")

# Load expanded 10+ page operational manual
with open(DATA_DIR / "turbomachinery_manual.md", "r", encoding="utf-8") as f:
    manual_text = f.read()

# Chunk into operational paragraphs
paragraphs = [p.strip() for p in manual_text.split("\n\n") if len(p.strip()) > 80]
print(f"[✓] Ingested {len(paragraphs)} technical operational paragraphs.")


[*] Verified Data Directory: C:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs\data\sample_documents
[✓] Ingested 27 technical operational paragraphs.


---
## Step 1: Query Transformation Strategies
### A) Multi-Query Expansion
Expands a single user question into 3 semantically diverse queries to overcome vocabulary mismatch.


In [2]:
import rag_universal_client as rc

def multi_query_expansion(query: str):
    """
    Generate multi-angle query expansions using Anthropic Claude (with Local Ollama backup).
    Transforms vague user queries into precise technical and emergency search angles.
    Zero canned mock fallbacks; raises error if models are offline.
    """
    prompt = (
        f"You are a technical search query expansion system for UST Global (UST) engineering operations.\n"
        f"Given the user query: '{query}', generate 3 diverse search query variations focusing on:\n"
        f"1. Standard technical terminology and component tags\n"
        f"2. Emergency and safety shutdown protocols\n"
        f"3. Operational limits and sensor parameters\n"
        f"Return ONLY the 3 queries, one per line. Do not number them or include explanations."
    )
    res = rc.call_rag_llm(prompt, temperature=0.3, max_tokens=150)
    print(f"  [Model Used: {res['tier_used']} | Latency: {res['latency_ms']} ms]")
    lines = [l.strip() for l in res["text"].split("\n") if l.strip() and not l.strip().startswith("#")]
    # Combine original query with diverse expansions
    expansions = [query] + [l for l in lines if l != query][:3]
    return expansions

test_query = "safety valve testing schedule"
print(f"[*] Expanding Query: '{test_query}'")
expanded_queries = multi_query_expansion(test_query)
print("\n[✓] Multi-Query Expansions:")
for idx, q in enumerate(expanded_queries, 1):
    print(f"  {idx}. {q}")


[*] Expanding Query: 'safety valve testing schedule'
  [Model Used: PRIMARY_CLOUD (claude-sonnet-4-6 - Live) | Latency: 2750.9 ms]

[✓] Multi-Query Expansions:
  1. safety valve testing schedule
  2. safety valve functional testing intervals PSV TSV SIL certification maintenance schedule
  3. emergency shutdown valve ESD testing protocol HIPPS safety instrumented system proof test procedure
  4. safety valve set pressure tolerance operating limits sensor calibration trip point verification


### B) Hypothetical Document Embeddings (HyDE)
Instead of searching with the raw user query, the LLM generates a hypothetical ideal answer. We then embed that hypothetical document to search the vector space!


In [3]:
import rag_universal_client as rc

def generate_hyde_document(query: str) -> str:
    """
    Generate a hypothetical engineering passage using Anthropic Claude (with Local Ollama backup).
    HyDE (Hypothetical Document Embeddings) synthesizes a realistic operational passage
    to bridge the semantic gap between vague questions and formal documentation.
    Zero canned mock fallbacks.
    """
    prompt = (
        f"You are a principal turbomachinery operations engineer at UST Global.\n"
        f"Write a dense, factual, 3-sentence technical manual passage that precisely answers this query:\n"
        f"'{query}'\n"
        f"Include precise numerical thresholds, equipment codes, or standard engineering practices."
    )
    res = rc.call_rag_llm(prompt, temperature=0.1, max_tokens=250)
    print(f"  [HyDE Model Used: {res['tier_used']} | Latency: {res['latency_ms']} ms]")
    return res["text"].strip()

hyde_query = "hydrostatic valve testing"
print(f"[*] Generating HyDE Passage for: '{hyde_query}'")
hyde_doc = generate_hyde_document(hyde_query)
print("\n[✓] Generated HyDE Hypothetical Document:")
print(f"  '{hyde_doc}'")


[*] Generating HyDE Passage for: 'hydrostatic valve testing'
  [HyDE Model Used: PRIMARY_CLOUD (claude-sonnet-4-6 - Live) | Latency: 6740.7 ms]

[✓] Generated HyDE Hypothetical Document:
  '## Hydrostatic Valve Testing – Technical Manual Passage

Hydrostatic valve testing shall be conducted in accordance with **API 598** (Valve Inspection and Testing) and **ASME B16.34**, wherein the shell/body test pressure is applied at **1.5× the rated working pressure (CWP)** for a minimum hold duration of **60 seconds** for valves ≤ NPS 2 and **120 seconds** for valves NPS 2½ through NPS 6, using a test medium of clean water (or water with corrosion inhibitor) at a maximum temperature of **52°C (125°F)**. Seat leakage acceptance criteria are classified per API 598 Table 4 and Table 5, where **Class VI soft-seated valves** must exhibit zero measurable leakage, while **Class IV metal-seated valves** permit a maximum leakage rate of **0.18 mL/min per inch of nominal pipe diameter** during the low-pre

---
## Step 2: Reciprocal Rank Fusion (RRF)
RRF merges ranking lists from different search algorithms without requiring normalized score calibration:
$$RRF\_Score(d) = \sum_{m \in Models} \frac{1}{k + rank_m(d)}$$
where $k$ is a constant (typically 60).


In [4]:
# 1. Lexical BM25 Index
tokenized_corpus = [re.findall(r'\w+', doc.lower()) for doc in paragraphs]
bm25 = BM25Okapi(tokenized_corpus)

# 2. Dense Vector Index
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = bi_encoder.encode(paragraphs, normalize_embeddings=True)

def hybrid_rrf_search(query, top_n=5, k=60):
    # Lexical Rank
    query_tokens = re.findall(r'\w+', query.lower())
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_ranking = np.argsort(bm25_scores)[::-1]
    
    # Dense Semantic Rank
    q_emb = bi_encoder.encode([query], normalize_embeddings=True)[0]
    dense_scores = np.dot(corpus_embeddings, q_emb)
    dense_ranking = np.argsort(dense_scores)[::-1]
    
    # Calculate RRF Scores
    rrf_scores = {}
    for rank, doc_idx in enumerate(bm25_ranking[:20]):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (1.0 / (k + rank + 1))
        
    for rank, doc_idx in enumerate(dense_ranking[:20]):
        rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + (1.0 / (k + rank + 1))
        
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return [(doc_idx, score, paragraphs[doc_idx]) for doc_idx, score in sorted_rrf]

rrf_results = hybrid_rrf_search("hydrostatic pressure testing intervals", top_n=3)
print("[✓] Top Reciprocal Rank Fusion (RRF) Chunks:")
for doc_idx, score, text in rrf_results:
    print(f"\n[Doc {doc_idx}] RRF Score: {score:.5f}")
    print(f"  {text[:180]}...")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10070.24it/s]


[✓] Top Reciprocal Rank Fusion (RRF) Chunks:

[Doc 19] RRF Score: 0.03252
  ## 8. Predictive Maintenance Cycles, Overhauls, and Hydrostatic Testing Protocols...

[Doc 22] RRF Score: 0.03202
  ### 8.2 Hydrostatic Pressure Integrity Verification Standard
Every **36 months**, the compressor outer barrel casing must undergo certified hydrostatic integrity verification:
1. A...

[Doc 8] RRF Score: 0.03126
  | Stage / Component | Parameter Description | Nominal Value | Alarm Level 1 (Warn) | Alarm Level 2 (Trip) | Monitoring System |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1st St...


---
## Step 3: Neural Cross-Encoder Re-Ranking
Now we pass the top candidates through a pre-trained Neural Cross-Encoder (`ms-marco-MiniLM-L-6-v2`) to re-score full cross-attention interaction.


In [7]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

query = "How often must compressor casing undergo hydrostatic pressure testing?"
candidate_chunks = [item[2] for item in rrf_results]

# Score (Query, Chunk) pairs
pairs = [[query, chunk] for chunk in candidate_chunks]
rerank_scores = cross_encoder.predict(pairs)

reranked = sorted(zip(rerank_scores, candidate_chunks), key=lambda x: x[0], reverse=True)

print(f"Query: '{query}'")
print("\n--- Cross-Encoder Re-Ranked Results ---")
for rank, (score, chunk) in enumerate(reranked, 1):
    print(f"\nRank {rank} (Logit Score: {score:+.4f}):")
    print(f"  {chunk}")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6898.85it/s]


Query: 'How often must compressor casing undergo hydrostatic pressure testing?'

--- Cross-Encoder Re-Ranked Results ---

Rank 1 (Logit Score: +8.1134):
  ### 8.2 Hydrostatic Pressure Integrity Verification Standard
Every **36 months**, the compressor outer barrel casing must undergo certified hydrostatic integrity verification:
1. All process nozzles must be blanked with ASME Class 900 certified blind flanges.
2. Casing filled with treated demineralized water containing corrosion inhibitor (UST-CHEM-CI-04).
3. Test pressure raised incrementally to **1.5x Maximum Allowable Working Pressure (MAWP)**.
4. Test pressure sustained continuously for **4.0 hours** minimum with calibrated digital pressure logging. Zero pressure drop allowed after temperature equalization.
5. All tests must be witnessed and certified by an independent Third-Party Inspection (TPI) agency.

Rank 2 (Logit Score: -5.8306):
  ## 8. Predictive Maintenance Cycles, Overhauls, and Hydrostatic Testing Protocols

Rank 3 (L

---
## 🏁 Production Engineering Takeaways & Exercises
1. **Never send first-pass vector results directly to the LLM:** Always filter top-25 results through a neural Cross-Encoder.
2. **Combine BM25 + Dense Vectors with RRF:** Lexical search guarantees exact serial number and code matching, while dense vectors guarantee conceptual comprehension.
